# Прогноз дохода клиентов

 - Задача прогнозирования дохода имеет особенное значение в Банке. 
Эта информация помогает точнее и более релевантно подбирать продукты и условия их приобретения, что в свою очередь вносит существенный вклад в прибыль Банка. 
 - Помимо ценности для Банка, оценка дохода является регуляторным требованием ЦБ в части расчета предельно допустимой кредитной нагрузки для клиента (далее ПДН)

# Импорт библиотек

In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Импорт функций

In [2]:
sys.path.append('../src') 

from preprocessing import (
    analyze_missing_columns,
    analyze_rows_statistics, 
    convert_basic_columns,
    analyze_realistic_selection,
    identify_problem_features,
    convert_numeric_strings,
    RobustAutoDataPreprocessor, 
    quick_preprocess,
    AdvancedSalaryEnricher
)

from data_explorer import DataExplorer

# Создание датафреймов

In [3]:
data_train = pd.read_csv('../data/hackathon_income_train.csv', sep=';')
data_test = pd.read_csv('../data/hackathon_income_test.csv', sep=';')
data_salary = pd.read_csv('../data/znr-2023.csv', sep=',')

# Первичный анализ данных

In [4]:
data_train.info(verbose='True', show_counts='True')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76786 entries, 0 to 76785
Data columns (total 224 columns):
 #    Column                                                                                           Non-Null Count  Dtype  
---   ------                                                                                           --------------  -----  
 0    id                                                                                               76786 non-null  int64  
 1    dt                                                                                               76786 non-null  object 
 2    target                                                                                           76786 non-null  object 
 3    turn_cur_cr_avg_act_v2                                                                           59078 non-null  object 
 4    salary_6to12m_avg                                                                                14875 non-null  object 
 

In [5]:
pd.set_option('display.max_columns', None)
data_train.head(5)

,id,dt,target,turn_cur_cr_avg_act_v2,salary_6to12m_avg,hdb_bki_total_max_limit,dp_ils_paymentssum_avg_12m,hdb_bki_total_cc_max_limit,incomeValue,gender,avg_cur_cr_turn,adminarea,turn_cur_cr_avg_v2,turn_cur_cr_max_v2,hdb_bki_total_pil_max_limit,age,dp_ils_avg_salary_1y,turn_cur_cr_sum_v2,by_category__amount__sum__eoperation_type_name__ishodjaschij_bystryj_platezh_sbp,turn_cur_db_sum_v2,turn_cur_db_avg_act_v2,dp_ils_avg_salary_2y,curr_rur_amt_cm_avg,turn_cur_db_avg_v2,by_category__amount__sum__eoperation_type_name__vhodjaschij_bystryj_platezh_sbp,dp_ils_paymentssum_avg_6m,avg_cur_db_turn,hdb_bki_active_cc_max_limit,incomeValueCategory,avg_by_category__amount__sum__cashflowcategory_name__vydacha_nalichnyh_v_bankomate,avg_credit_turn_rur,dp_ils_salary_ratio_1y3y,by_category__amount__sum__eoperation_type_name__perevod_po_nomeru_telefona,turn_cur_cr_7avg_avg_v2,dp_ils_accpayment_avg_12m,curbal_usd_amt_cm_avg,avg_by_category__amount__sum__cashflowcategory_name__supermarkety,avg_loan_cnt_with_insurance,avg_by_category__amount__sum__cashflowcategory_name__gipermarkety,city_smart_name,uniV5,turn_cur_db_max_v2,avg_by_category__amount__sum__cashflowcategory_name__kafe,turn_other_db_max_v2,turn_cur_cr_min_v2,hdb_bki_other_active_pil_outstanding,dp_ewb_last_employment_position,turn_cur_db_min_v2,hdb_bki_total_products,per_capita_income_rur_amt,avg_debet_turn_rur,hdb_relend_active_max_psk,dda_rur_amt_curr_v2,mob_cnt_days,dp_ils_days_from_last_doc,avg_6m_money_transactions,transaction_category_supermarket_percent_cnt_2m,pil,hdb_bki_total_max_overdue_sum,avg_6m_clothing,avg_by_category__amount__sum__cashflowcategory_name__elektronnye_dengi,addrref,bki_total_auto_cnt,dp_payoutincomedata_payout_avg_3_month,hdb_outstand_sum,avg_3m_money_transactions,dp_address_unique_regions,min_balance_rur_amt_6m_af,transaction_category_supermarket_sum_cnt_m3_4,dp_payoutincomedata_payout_max_3_month,hdb_bki_total_ip_max_limit,hdb_bki_total_cnt,blacklist_flag,bki_total_oth_cnt,dp_payoutincomedata_payout_sum_3_month,hdb_relend_outstand_sum,total_rur_amt_cm_avg,mob_cover_days,dp_payoutincomedata_payout_max_6_month,label_Below_50k_share_r1,turn_fdep_db_sum_v2,dp_ils_accpayment_avg_6m_current,transaction_category_cash_percent_amt_2m,curr_rur_amt_3m_avg,transaction_category_restaurants_sum_amt_m2,loan_cnt,turn_fdep_db_avg_v2,turn_cur_db_7avg_avg_v2,bki_total_ip_max_outstand,amount_by_category_90d__summarur_amt__sum__cashflowcategory_name__vydacha_nalichnyh_v_bankomate,profit_income_out_rur_amt_12m,avg_6m_hotels,hdb_ovrd_sum,dp_ils_total_seniority,dp_ils_paymentssum_avg_6m_current,smsInWavg6m,avg_fdep_db_turn,device_iphone_avg,by_category__amount__sum__eoperation_type_name__platezh_za_mobilnyj_cherez_ps,avg_balance_rur_amt_1m_af,curr_rur_amt_cm_avg_period_days_ago_v2,avg_by_category__amount__sum__cashflowcategory_name__oteli,hdb_bki_total_ip_cnt,hdb_bki_active_cc_max_outstand,hdb_other_outstand_sum,days_to_last_transaction,hdb_bki_total_pil_max_overdue,vert_pil_last_credit_step_screen_view_3m,acard,bki_total_il_max_limit,other_credits_count,tz_msk_timedelta,turn_save_db_min_v2,profit_income_out_rur_amt_9m,dp_ils_ipkcurrentyear_currentyearpensfactor,avg_by_category__amount__sum__cashflowcategory_name__odezhda,cntOnnRinCallAvg6m,dda_rur_amt_3m_avg,winback_cnt,salary_median_in_gex_r1,dp_payoutincomedata_payout_avg_prev_year,avg_amount_daily_transactions_90d,vert_has_app_ru_tinkoff_investing,transaction_category_supermarket_inc_cnt_2m,vert_pil_sms_success_3m,min_balance_rur_amt_1m_af,dp_ils_max_seniority,avg_by_category__amount__sum__cashflowcategory_name__set_supermarketov,label_500k_to_1M_share_r1,avg_by_category__amount__sum__cashflowcategory_name__zarubezhnye_finansovye_operatsii,bki_total_products,avg_6m_all,dp_ils_avg_simultanious_jobs_5y,dp_ewb_dismissal_due_contract_violation_by_lb_cnt,summarur_1m_purch,diff_avg_cr_db_turn,dp_ils_cnt_changes_1y,dp_ils_employeers_cnt_last_month,dp_payoutincomedata_payout_avg_6_month,dp_ewb_last_organization,by_category__amount__sum_

In [6]:
data_test.info(verbose='True', show_counts='True')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73214 entries, 0 to 73213
Data columns (total 222 columns):
 #    Column                                                                                           Non-Null Count  Dtype  
---   ------                                                                                           --------------  -----  
 0    id                                                                                               73214 non-null  int64  
 1    dt                                                                                               73214 non-null  object 
 2    turn_cur_cr_avg_act_v2                                                                           57244 non-null  object 
 3    salary_6to12m_avg                                                                                7634 non-null   object 
 4    hdb_bki_total_max_limit                                                                          62809 non-null  float64
 

In [7]:
data_test.head(5)

,id,dt,turn_cur_cr_avg_act_v2,salary_6to12m_avg,hdb_bki_total_max_limit,dp_ils_paymentssum_avg_12m,hdb_bki_total_cc_max_limit,incomeValue,gender,avg_cur_cr_turn,adminarea,turn_cur_cr_avg_v2,turn_cur_cr_max_v2,hdb_bki_total_pil_max_limit,age,dp_ils_avg_salary_1y,turn_cur_cr_sum_v2,by_category__amount__sum__eoperation_type_name__ishodjaschij_bystryj_platezh_sbp,turn_cur_db_sum_v2,turn_cur_db_avg_act_v2,dp_ils_avg_salary_2y,curr_rur_amt_cm_avg,turn_cur_db_avg_v2,by_category__amount__sum__eoperation_type_name__vhodjaschij_bystryj_platezh_sbp,dp_ils_paymentssum_avg_6m,avg_cur_db_turn,hdb_bki_active_cc_max_limit,incomeValueCategory,avg_by_category__amount__sum__cashflowcategory_name__vydacha_nalichnyh_v_bankomate,avg_credit_turn_rur,dp_ils_salary_ratio_1y3y,by_category__amount__sum__eoperation_type_name__perevod_po_nomeru_telefona,turn_cur_cr_7avg_avg_v2,dp_ils_accpayment_avg_12m,curbal_usd_amt_cm_avg,avg_by_category__amount__sum__cashflowcategory_name__supermarkety,avg_loan_cnt_with_insurance,avg_by_category__amount__sum__cashflowcategory_name__gipermarkety,city_smart_name,uniV5,turn_cur_db_max_v2,avg_by_category__amount__sum__cashflowcategory_name__kafe,turn_other_db_max_v2,turn_cur_cr_min_v2,hdb_bki_other_active_pil_outstanding,dp_ewb_last_employment_position,turn_cur_db_min_v2,hdb_bki_total_products,per_capita_income_rur_amt,avg_debet_turn_rur,hdb_relend_active_max_psk,dda_rur_amt_curr_v2,mob_cnt_days,dp_ils_days_from_last_doc,avg_6m_money_transactions,transaction_category_supermarket_percent_cnt_2m,pil,hdb_bki_total_max_overdue_sum,avg_6m_clothing,avg_by_category__amount__sum__cashflowcategory_name__elektronnye_dengi,addrref,bki_total_auto_cnt,dp_payoutincomedata_payout_avg_3_month,hdb_outstand_sum,avg_3m_money_transactions,dp_address_unique_regions,min_balance_rur_amt_6m_af,transaction_category_supermarket_sum_cnt_m3_4,dp_payoutincomedata_payout_max_3_month,hdb_bki_total_ip_max_limit,hdb_bki_total_cnt,blacklist_flag,bki_total_oth_cnt,dp_payoutincomedata_payout_sum_3_month,hdb_relend_outstand_sum,total_rur_amt_cm_avg,mob_cover_days,dp_payoutincomedata_payout_max_6_month,label_Below_50k_share_r1,turn_fdep_db_sum_v2,dp_ils_accpayment_avg_6m_current,transaction_category_cash_percent_amt_2m,curr_rur_amt_3m_avg,transaction_category_restaurants_sum_amt_m2,loan_cnt,turn_fdep_db_avg_v2,turn_cur_db_7avg_avg_v2,bki_total_ip_max_outstand,amount_by_category_90d__summarur_amt__sum__cashflowcategory_name__vydacha_nalichnyh_v_bankomate,profit_income_out_rur_amt_12m,avg_6m_hotels,hdb_ovrd_sum,dp_ils_total_seniority,dp_ils_paymentssum_avg_6m_current,smsInWavg6m,avg_fdep_db_turn,device_iphone_avg,by_category__amount__sum__eoperation_type_name__platezh_za_mobilnyj_cherez_ps,avg_balance_rur_amt_1m_af,curr_rur_amt_cm_avg_period_days_ago_v2,avg_by_category__amount__sum__cashflowcategory_name__oteli,hdb_bki_total_ip_cnt,hdb_bki_active_cc_max_outstand,hdb_other_outstand_sum,days_to_last_transaction,hdb_bki_total_pil_max_overdue,vert_pil_last_credit_step_screen_view_3m,acard,bki_total_il_max_limit,other_credits_count,tz_msk_timedelta,turn_save_db_min_v2,profit_income_out_rur_amt_9m,dp_ils_ipkcurrentyear_currentyearpensfactor,avg_by_category__amount__sum__cashflowcategory_name__odezhda,cntOnnRinCallAvg6m,dda_rur_amt_3m_avg,winback_cnt,salary_median_in_gex_r1,dp_payoutincomedata_payout_avg_prev_year,avg_amount_daily_transactions_90d,vert_has_app_ru_tinkoff_investing,transaction_category_supermarket_inc_cnt_2m,vert_pil_sms_success_3m,min_balance_rur_amt_1m_af,dp_ils_max_seniority,avg_by_category__amount__sum__cashflowcategory_name__set_supermarketov,label_500k_to_1M_share_r1,avg_by_category__amount__sum__cashflowcategory_name__zarubezhnye_finansovye_operatsii,bki_total_products,avg_6m_all,dp_ils_avg_simultanious_jobs_5y,dp_ewb_dismissal_due_contract_violation_by_lb_cnt,summarur_1m_purch,diff_avg_cr_db_turn,dp_ils_cnt_changes_1y,dp_ils_employeers_cnt_last_month,dp_payoutincomedata_payout_avg_6_month,dp_ewb_last_organization,by_category__amount__sum__eopera

In [8]:
print("Cтолбцы для удаления (>50%):")
columns_to_drop = analyze_missing_columns(data_train,
                                          threshold=0.5,
                                          sort_alphabetical=True)

Cтолбцы для удаления (>50%):
Столбцы с пропусками > 50%:
amount_by_category_90d__summarur_amt__sum__cashflowcategory_name__elektronnye_dengi: 67.3% (51642 пропусков)
amount_by_category_90d__summarur_amt__sum__cashflowcategory_name__vydacha_nalichnyh_v_bankomate: 62.0% (47645 пропусков)
avg_balance_rur_amt_1m_af: 85.5% (65653 пропусков)
avg_by_category__amount__sum__cashflowcategory_name__kosmetika: 62.0% (47632 пропусков)
avg_by_category__amount__sum__cashflowcategory_name__odezhda: 54.2% (41616 пропусков)
avg_by_category__amount__sum__cashflowcategory_name__oteli: 88.5% (67925 пропусков)
avg_by_category__amount__sum__cashflowcategory_name__platezhi_cherez_internet: 98.4% (75589 пропусков)
avg_by_category__amount__sum__cashflowcategory_name__puteshestvija: 86.6% (66512 пропусков)
avg_by_category__amount__sum__cashflowcategory_name__reklama_v_internete: 93.2% (71564 пропусков)
avg_by_category__amount__sum__cashflowcategory_name__set_supermarketov: 87.5% (67217 пропусков)
avg_by_category

Удаляем столбцы из обоих наборов данных

In [9]:
all_columns_to_drop = list(columns_to_drop) + ['dt', 'avg_3m_no_cat', 'bki_active_auto_cnt',
                                              'bki_total_active_products', 'cntBlockWavg6m', 'days_after_last_request', 
                                              'device_iphone_avg', 'hdb_bki_active_pil_cnt', 'hdb_bki_last_product_days',
                                              'hdb_bki_total_cnt', 'hdb_bki_total_pil_max_del90', 'hdb_bki_total_products',
                                              'hdb_other_outstand_sum', 'tz_msk_timedelta', 'winback_cnt']

In [10]:
existing_columns_train = [col for col in all_columns_to_drop if col in data_train.columns]
existing_columns_test = [col for col in all_columns_to_drop if col in data_test.columns]

data_train_clean = data_train.drop(columns=existing_columns_train)
data_test_clean = data_test.drop(columns=existing_columns_test)

print(f"\nУдалено столбцов: {len(all_columns_to_drop)}")
print(f"Новый размер обучающих данных: {data_train_clean.shape}")
print(f"Новый размер тестовых данных: {data_test_clean.shape}")


Удалено столбцов: 98
Новый размер обучающих данных: (76786, 126)
Новый размер тестовых данных: (73214, 124)


In [11]:
print("Cтроки для удаления (>50%):")
rows_to_drop_ids = analyze_rows_statistics(data_train_clean)

Cтроки для удаления (>50%):
СТАТИСТИКА ПО СТРОКАМ:
Всего строк: 76786
Всего столбцов: 126
Максимально возможное пропусков в строке: 126

Распределение пропусков по строкам:
Строк без пропусков: 7340 (9.6%)
Строк с 50%+ пропусками: 13227 (17.2%)
Строк с 90%+ пропусками: 0 (0.0%)

Строк для удаления (порог 50%): 13227


In [12]:
existing_rows_train = [id for id in rows_to_drop_ids if id in data_train_clean.index]
data_train_clean = data_train_clean.drop(index=existing_rows_train)

# Изменение типов и заполнение пропусков

Исправим id, w, target

In [13]:
data_train_clean = convert_basic_columns(data_train_clean)
data_test = convert_basic_columns(data_test)

In [14]:
analyze_realistic_selection(data_train_clean)

Всего клиентов: 76672

ЗАПОЛНЕННОСТЬ ПРИЗНАКОВ:
  turn_cur_cr_avg_act_v2: 58986 (76.9%)
  hdb_bki_total_max_limit: 67316 (87.8%)
  hdb_bki_total_cc_max_limit: 62951 (82.1%)
  incomeValue: 63714 (83.1%)
  gender: 76672 (100.0%)
  avg_cur_cr_turn: 60120 (78.4%)
  adminarea: 57136 (74.5%)
  turn_cur_cr_avg_v2: 59340 (77.4%)
  turn_cur_cr_max_v2: 59340 (77.4%)
  hdb_bki_total_pil_max_limit: 61912 (80.7%)
  age: 76672 (100.0%)
  turn_cur_cr_sum_v2: 59340 (77.4%)
  turn_cur_db_sum_v2: 59340 (77.4%)
  turn_cur_db_avg_act_v2: 58489 (76.3%)
  curr_rur_amt_cm_avg: 58692 (76.5%)
  turn_cur_db_avg_v2: 59340 (77.4%)
  avg_cur_db_turn: 60120 (78.4%)
  hdb_bki_active_cc_max_limit: 56790 (74.1%)
  incomeValueCategory: 63714 (83.1%)
  avg_by_category__amount__sum__cashflowcategory_name__vydacha_nalichnyh_v_bankomate: 41822 (54.5%)
  avg_credit_turn_rur: 60120 (78.4%)
  by_category__amount__sum__eoperation_type_name__perevod_po_nomeru_telefona: 38596 (50.3%)
  turn_cur_cr_7avg_avg_v2: 59340 (77.4%)
  cu

Анализируем проблемы

In [15]:
problems = identify_problem_features(data_train_clean)


⚠️  ПРОБЛЕМНЫЕ ПРИЗНАКИ ДЛЯ ОБРАБОТКИ:
Высокие пропуски (>30%): 24
   - avg_by_category__amount__sum__cashflowcategory_name__vydacha_nalichnyh_v_bankomate: 45.5% пропусков
   - by_category__amount__sum__eoperation_type_name__perevod_po_nomeru_telefona: 49.7% пропусков
   - avg_by_category__amount__sum__cashflowcategory_name__supermarkety: 37.8% пропусков
   - avg_by_category__amount__sum__cashflowcategory_name__gipermarkety: 41.3% пропусков
   - avg_by_category__amount__sum__cashflowcategory_name__kafe: 38.9% пропусков
   - turn_other_db_max_v2: 37.4% пропусков
   - transaction_category_supermarket_percent_cnt_2m: 44.8% пропусков
   - avg_by_category__amount__sum__cashflowcategory_name__elektronnye_dengi: 48.3% пропусков
   - transaction_category_supermarket_sum_cnt_m3_4: 47.5% пропусков
   - smsInWavg6m: 32.3% пропусков
   - bki_total_il_max_limit: 38.5% пропусков
   - avg_amount_daily_transactions_90d: 32.6% пропусков
   - transaction_category_supermarket_inc_cnt_2m: 44.8% пропусков

Преобразование типов данных

In [16]:
train_processed = convert_numeric_strings(data_train_clean.copy(), problems)
test_processed = convert_numeric_strings(data_test.copy(), problems)

In [17]:
train_processed, test_processed = quick_preprocess(train_processed, test_processed)

print(f"Тренировочные: {train_processed.shape}")
print(f"Тестовые: {test_processed.shape}")

Быстрая автоматическая обработка
Анализ данных

Отчет об анализе данных:
Числовые колонки: 20
   - hdb_bki_total_max_limit: float64, 26586 уникальных
   - hdb_bki_total_cc_max_limit: float64, 6382 уникальных
   - hdb_bki_total_pil_max_limit: float64, 27609 уникальных
   - hdb_bki_active_cc_max_limit: float64, 4479 уникальных
   - hdb_bki_total_max_overdue_sum: float64, 32962 уникальных
   - bki_total_auto_cnt: float64, 15 уникальных
   - bki_total_oth_cnt: float64, 24 уникальных
   - hdb_bki_total_ip_cnt: float64, 16 уникальных
   - hdb_bki_active_cc_max_outstand: float64, 42446 уникальных
   - hdb_bki_total_pil_max_overdue: float64, 29161 уникальных
   - bki_total_il_max_limit: float64, 25072 уникальных
   - bki_total_products: float64, 8 уникальных
   - bki_total_max_limit: float64, 23820 уникальных
   - hdb_bki_total_active_products: float64, 88 уникальных
   - hdb_bki_total_micro_cnt: float64, 149 уникальных
   - hdb_bki_total_cc_max_overdue: float64, 20756 уникальных
   - hdb_bki_

/Users/dataalph/Desktop/da_venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/dataalph/Desktop/da_venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [18]:
print("\n📈 Статистика после обработки:")
print(train_processed.info())


📈 Статистика после обработки:
<class 'pandas.core.frame.DataFrame'>
Index: 76672 entries, 114 to 76785
Columns: 126 entries, id to w
dtypes: float64(121), int32(1), int8(1), object(3)
memory usage: 73.5+ MB
None


Добавление зарплат по регионам

In [19]:
enricher = AdvancedSalaryEnricher(data_salary, region_col='Среднемесячная зп', salary_col='2023')
train_processed = enricher.fit_transform(train_processed)
test_processed = enricher.fit_transform(test_processed)


 СТАТИСТИКА МАТЧИНГА ЗАРПЛАТ:
 Найдены зарплаты для: 76672/76672 (100.0%)
 Диапазон зарплат: 28931 - 110771
 Средняя зарплата: 59504

 СТАТИСТИКА МАТЧИНГА ЗАРПЛАТ:
 Найдены зарплаты для: 73214/73214 (100.0%)
 Диапазон зарплат: 28931 - 110771
 Средняя зарплата: 59072


In [20]:
train_processed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 76672 entries, 114 to 76785
Columns: 124 entries, id to salary
dtypes: float64(122), int32(1), int8(1)
memory usage: 72.3 MB


# Сохранение данных

In [21]:
train_processed.to_pickle('../data/df_train.pkl')
test_processed.to_pickle('../data/df_test.pkl')